# Intervention / program effectiveness (Hiraya Haven)

**Pipeline:** Explore whether intervention plan categories/services are associated with improved education or health trajectories.

**Data:** `intervention_plans`, `education_records`, `health_wellbeing_records` from `../Data/hiraya.db`.

**Explanatory:** associations between plan categories and outcomes, controlling for resident baseline.

**Predictive:** predict next-month education progress from current plan mix + recent history.

> Causal caution: This is observational data; interventions are not randomly assigned.

> **IS455 pipeline structure:** (1) Problem framing → (2) Data acquisition, preparation & exploration → (3) Modeling → (4) Evaluation & selection → (5) Feature selection & interpretation → (6) Explanatory analysis (associations; not causal proof) → (7) Deployment.

## 1. Problem framing

Hiraya Haven wants to understand which interventions appear to help residents progress (education/health) and how to prioritize services.

- **Predictive goal:** predict next-month education progress.
- **Explanatory goal:** examine associations between plan category mix and outcomes.

**Success metrics:** predictive MAE; explanatory coherence + limitations.

## 2. Data acquisition, preparation & exploration

Data comes from `intervention_plans`, `education_records`, `health_wellbeing_records` in `../Data/hiraya.db`. We build resident-month outcomes and merge plan category counts.

## 3. Modeling

RandomForestRegressor for predictive performance; feature importances for a first-pass “what matters.”

## 4. Evaluation & selection

Report MAE on a future time split. Discuss whether performance is sufficient for operational use vs just exploration.

## 5. Feature selection & interpretation

Compare **RandomForest `feature_importances_`** for the trajectory-only baseline vs. the add-on with **cumulative** plan-by-month features.

## 6. Explanatory analysis (associations; not causal proof)

Interventions are assigned based on need (selection bias). Treat category associations as descriptive/hypothesis-generating.

## 7. Deployment

Deploy as a reporting/dashboard module first; only automate decisions after collecting better causal covariates (staffing, severity baselines).

In [4]:
# ============================================================================
# Intervention effectiveness — Baseline: education progress vs plan categories
# ============================================================================
# Notebook code cell overview — see inline comments below.
#
# Merges education + health resident-month panels with STATIC plan counts per resident.
#   (Counts all plans ever — can leak future information into early months; add-on cell fixes this.)
# Target progress_next is next month's mean education progress_percent within resident.
# RandomForest + time-based train/test split (by calendar month) evaluates forecasting sanity.
# Feature importances are associative; plans are not randomly assigned — causal claims need design.
#
from __future__ import annotations

import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

sns.set_theme(style="whitegrid", context="notebook")

ROOT = Path("..").resolve()
DB_PATH = ROOT / "Data" / "hiraya.db"

with sqlite3.connect(DB_PATH) as conn:
    plans = pd.read_sql_query('SELECT * FROM "intervention_plans"', conn)
    edu = pd.read_sql_query('SELECT * FROM "education_records"', conn)
    health = pd.read_sql_query('SELECT * FROM "health_wellbeing_records"', conn)

edu["record_date"] = pd.to_datetime(edu["record_date"], errors="coerce")
health["record_date"] = pd.to_datetime(health["record_date"], errors="coerce")

edu["month"] = edu["record_date"].dt.to_period("M").dt.to_timestamp()
health["month"] = health["record_date"].dt.to_period("M").dt.to_timestamp()

# Plan features: counts by resident (static) and category mix
plans["plan_category"] = plans["plan_category"].fillna("(unknown)")
plan_counts = plans.groupby(["resident_id", "plan_category"], as_index=False).size().rename(columns={"size": "n_plans"})
plan_wide = plan_counts.pivot_table(index="resident_id", columns="plan_category", values="n_plans", fill_value=0)
plan_wide.columns = [f"plans_cat_{c}" for c in plan_wide.columns]
plan_wide = plan_wide.reset_index()

# Resident-month panel for education outcome
edu_panel = edu.groupby(["resident_id", "month"], as_index=False).agg(
    progress_percent=("progress_percent", "mean"),
    attendance_rate=("attendance_rate", "mean"),
)

health_panel = health.groupby(["resident_id", "month"], as_index=False).agg(
    general_health_score=("general_health_score", "mean"),
)

panel = edu_panel.merge(health_panel, on=["resident_id", "month"], how="outer")
panel = panel.merge(plan_wide, on="resident_id", how="left")

# Fill plan features with 0
for c in panel.columns:
    if c.startswith("plans_cat_"):
        panel[c] = panel[c].fillna(0)

panel = panel.sort_values(["resident_id", "month"]).copy()
g = panel.groupby("resident_id", group_keys=False)
panel["progress_next"] = g["progress_percent"].shift(-1)

# Modeling dataset
model = panel.dropna(subset=["progress_percent", "progress_next", "month"]).copy()

# Omit static plans_cat_* here — they aggregate all future plans per resident (leakage). See add-on for cumulative counts.
feature_cols = [
    "progress_percent",
    "attendance_rate",
    "general_health_score",
]

X = model[feature_cols].fillna(0)
y = model["progress_next"].astype(float)

# Time split
months = sorted(model["month"].unique())
cut = months[int(0.75 * len(months))]
train = model["month"] < cut

rf = RandomForestRegressor(n_estimators=300, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X[train], y[train])
pred = rf.predict(X[~train])
mae = mean_absolute_error(y[~train], pred)
print("MAE (next-month progress; no static plan features):", round(mae, 3))

imp = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
imp.head(15)

MAE (predict next-month education progress): 4.798


progress_percent             0.911345
attendance_rate              0.045740
general_health_score         0.042915
plans_cat_Education          0.000000
plans_cat_Physical Health    0.000000
plans_cat_Safety             0.000000
dtype: float64

### Recap (sections 6–7)

- Plan categories are **not randomly assigned**; higher-risk residents may receive more intensive plans.
- Any positive/negative association between plan categories and outcomes could be due to **selection**, not plan efficacy.

Treat this pipeline as a starting point to identify which interventions deserve deeper review or better measurement.

**Deployment ideas:** reporting view for outcomes by plan category mix; stronger causal designs if staffing/severity data are added.

## Add-on: leakage-reduced plan features

The baseline model uses overall plan counts by resident, which can leak “future” plans into earlier months. Below we rebuild plan-category features as **cumulative counts up to month t** for each resident-month, then re-fit the predictive model.

In [5]:
# ============================================================================
# Add-on — Cumulative plan counts by month (leakage-reduced plan features)
# ============================================================================
# Notebook code cell overview — see inline comments below.
#
# Uses plan created_at (or case_conference_date) to place plans in the month they started.
# Builds resident×month counts, then cumsum within resident so features only use past/current plans.
# Drops static plans_cat_* from `model` before join — same column names would collide in pandas.merge.
# Re-fit RF; compare MAE to baseline to see how much 'when' the plan appeared matters.
#
# Rebuild plan features by month using plan created_at (if available)

plans2 = plans.copy()
plans2["created_at"] = pd.to_datetime(plans2.get("created_at"), errors="coerce")
plans2["plan_month"] = plans2["created_at"].dt.to_period("M").dt.to_timestamp()

# If created_at is missing everywhere, fall back to case_conference_date, else use plan_month
if plans2["plan_month"].isna().all() and "case_conference_date" in plans2.columns:
    plans2["case_conference_date"] = pd.to_datetime(plans2["case_conference_date"], errors="coerce")
    plans2["plan_month"] = plans2["case_conference_date"].dt.to_period("M").dt.to_timestamp()

# Build cumulative counts per resident over months
plan_events = plans2.dropna(subset=["resident_id", "plan_month"]).copy()
plan_events["plan_category"] = plan_events["plan_category"].fillna("(unknown)")

# Expand to resident-month panel months
months_all = pd.Series(sorted(panel["month"].dropna().unique()), name="month")
res_month = panel[["resident_id", "month"]].dropna().drop_duplicates().sort_values(["resident_id", "month"])

# Monthly plan counts then cumulative
monthly_counts = (
    plan_events.groupby(["resident_id", "plan_month", "plan_category"], as_index=False)
    .size()
    .rename(columns={"size": "n"})
    .rename(columns={"plan_month": "month"})
)

wide = (
    monthly_counts.pivot_table(index=["resident_id", "month"], columns="plan_category", values="n", fill_value=0)
    .sort_index()
)

# Reindex to include all resident-month rows, then cumulative sum within resident
wide = wide.reindex(res_month.set_index(["resident_id", "month"]).index, fill_value=0)
wide_cum = wide.groupby(level=0).cumsum()
wide_cum.columns = [f"plans_cat_{c}" for c in wide_cum.columns]

# Baseline `model` already has static plans_cat_*; drop before joining cumulative counts
model2 = model.drop(columns=[c for c in model.columns if c.startswith("plans_cat_")]).set_index(
    ["resident_id", "month"]
)
model2 = model2.join(wide_cum, how="left")
for c in model2.columns:
    if c.startswith("plans_cat_"):
        model2[c] = model2[c].fillna(0)

feature_cols2 = [
    "progress_percent",
    "attendance_rate",
    "general_health_score",
] + [c for c in model2.columns if c.startswith("plans_cat_")]

X2 = model2[feature_cols2].fillna(0)
y2 = model2["progress_next"].astype(float)

# Time split
months = sorted(model2.index.get_level_values("month").unique())
cut = months[int(0.75 * len(months))]
train = model2.index.get_level_values("month") < cut

rf2 = RandomForestRegressor(n_estimators=300, max_depth=10, random_state=42, n_jobs=-1)
rf2.fit(X2[train], y2[train])
pred2 = rf2.predict(X2[~train])
mae2 = mean_absolute_error(y2[~train], pred2)
print("MAE (leakage-reduced plan features):", round(mae2, 3))

imp2 = pd.Series(rf2.feature_importances_, index=feature_cols2).sort_values(ascending=False)
imp2.head(15)

MAE (leakage-reduced plan features): 4.798


progress_percent             0.911345
attendance_rate              0.045740
general_health_score         0.042915
plans_cat_Education          0.000000
plans_cat_Physical Health    0.000000
plans_cat_Safety             0.000000
dtype: float64